In [23]:
%load_ext autoreload
%autoreload all

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [24]:
import pickle
import itertools

import polars as pl
import numpy as np
from tqdm import tqdm


import src.graph_fct as graph_fct
import src.utils as ut
from src.tokenizer import GraphTokenizer
from src.iterative_selection import iterative_approach_w_graph_red, IterativeMarginalGainMax
import src.configs_fd as configs


In [25]:
prim_candidate_only  = True

In [26]:
df_concept_hug = pl.read_parquet(configs.GraphConfig.concept_path)

fd_list_1 = (
            pl.read_parquet(configs.GraphConfig().official_release_path)
                .filter(pl.col("status") == "defined")["id"].unique().to_list()
            )
fd_list_2 = (
            df_concept_hug.filter(pl.col("concept_type") == "SCT_POST")["id"].to_list()
            )
fd_list = fd_list_1 + fd_list_2

prim_list = (
            pl.read_parquet(configs.GraphConfig().official_release_path)
                .filter(pl.col("status") == "primitive")["id"].unique().to_list()
            )
df_concept_hug = (
            df_concept_hug
            .with_columns(is_mapped = pl.col("id").is_in(fd_list))
                )

In [27]:
df_relation = pl.read_parquet(configs.GraphConfig.relation_path)
g_is_a = graph_fct.get_is_a_graph(df_relation)

df_concept_hug_w_depth = graph_fct.get_max_min_depth_snomed(G=g_is_a, df_all_concepts=df_concept_hug)

concept_propagate = ut.ConceptPropagate(df_relation, df_concept_hug_w_depth)
mapped_candidate_rel_dist_prop_full = concept_propagate.build_all_distance_and_relations()

if prim_candidate_only:
    mapped_candidate_rel_dist_prop_full = (mapped_candidate_rel_dist_prop_full
                                           .filter(pl.col("dst.id").is_in(prim_list)))
    

candidate_reachable_child_map = ut.precompute_candidate_reachable_child_map(
    g_is_a,
    mapped_candidate_rel_dist_prop_full["dst.id"].unique(),
)

# Shared filtered view used by steps 2-4
mapped_candidate_rel_dist_prop = (
    mapped_candidate_rel_dist_prop_full
    .filter(pl.col("distance") <= configs.TokenizerParam().max_dist_candidate)
)


Precomputing reachable child candidates: 100%|██████████| 42469/42469 [00:10<00:00, 4101.44it/s]


In [28]:
candidates_set = set(mapped_candidate_rel_dist_prop_full["dst.id"].unique().to_list())
len(candidates_set)

42469

# check if all candidates are legitime

In [29]:
A = set(pl.read_parquet(f"{configs.Baselines().path}highest_degree.parquet")["dst.id"].unique().to_list()) 
set(A).issubset(candidates_set)


True

In [30]:
A = set(pl.read_parquet(f"{configs.Baselines().path}highest_degree_dist_1.parquet")["dst.id"].unique().to_list()) 
set(A).issubset(candidates_set)


True

In [31]:
A = set(pl.read_parquet(f"{configs.Baselines().path}most_children.parquet")["candidate"].unique().to_list()) 
set(A).issubset(candidates_set)

True

In [32]:
A = set(pl.read_parquet(f"{configs.IterativeMarginalGain().path}_inv_exp.parquet")["candidate_id"].unique().to_list()) 
set(A).issubset(candidates_set)

True

In [33]:
A = set(pl.read_parquet(f"{configs.IterativeMarginalGain().path}_inv.parquet")["candidate_id"].unique().to_list()) 
set(A).issubset(candidates_set)

True

In [34]:
A = set(pl.read_parquet(f"{configs.IterativeGraphRed().path}sum_inv.parquet")["candidate"].unique().to_list()) 
set(A).issubset(candidates_set)

True

In [35]:
A = set(pl.read_parquet(f"{configs.IterativeGraphRed().path}sum_inv_exp.parquet")["candidate"].unique().to_list()) 
set(A).issubset(candidates_set)

True

In [36]:
A = set(pl.read_parquet(f"{configs.IterativeGraphRed().path}tempered_sum_inv.parquet")["candidate"].unique().to_list()) 
set(A).issubset(candidates_set)

True

In [37]:
A = set(pl.read_parquet(f"{configs.IterativeGraphRed().path}tempered_sum_inv_exp.parquet")["candidate"].unique().to_list()) 
set(A).issubset(candidates_set)

True

# check why sem cov > 1

In [38]:
tokenizer = GraphTokenizer(
    df_concept_hug_w_depth,
    mapped_candidate_rel_dist_prop,
    df_relation,
    candidate_reachable_child_map,
    configs.TokenizerParam().max_dist_candidate,
)


In [39]:
k = 11500
ordered_ids = pl.read_parquet(f"{configs.IterativeMarginalGain().path}_inv.parquet")["candidate_id"].to_list()

In [40]:
scores, results, df_tok_all_n_dist = tokenizer.evaluate_components_and_tokenize(ordered_ids[:k])


In [41]:
df_tok_all_n_dist.filter(pl.col("mapped_id") == "234166002")["relation"].n_unique()

7

In [42]:
tokenizer.sem_cov_base.filter(pl.col("mapped_id") == "234166002")

mapped_id,num_relation_type
str,u32
"""234166002""",7


In [43]:
(df_relation
 .filter(pl.col("src.id").is_in(tokenizer.mapped_id_list))
 .group_by("src.id")
            .agg(pl.col("relation").n_unique().alias("num_relation_type"))
            .rename({"src.id": "mapped_id"})
            .filter(pl.col("mapped_id") == "234166002")
            )

mapped_id,num_relation_type
str,u32
"""234166002""",7


In [44]:
scores

{'sem_cov_score': 0.8350416154728652,
 'distance_score': 0.6467967855730752,
 'uniqueness_entropy_score': 0.9271057322450296,
 'conciseness_score': 0.2973122989574208,
 'compression_rate': 0.06456093391515588,
 'UNK_rate': 0.0012202528228890189,
 'exact_rate': 0.0,
 'num_candidates': 11481}